# Download data

!mkdir -p /home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813

!wget -q -P /home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813  \
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE123nnn/GSE123813/suppl/GSE123813_bcc_all_metadata.txt.gz

!wget -q -P /home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813 \
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE123nnn/GSE123813/suppl/GSE123813_bcc_scRNA_counts.txt.gz

In [ ]:
# preprocess BCC 

import pandas as pd
import scanpy as sc

DATA_DIR = "/home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813"

# load metadata
metadata = pd.read_csv(
    f"{DATA_DIR}/GSE123813_bcc_all_metadata.txt.gz",
    sep="\t"
)

# load counts
counts = pd.read_csv(
    f"{DATA_DIR}/GSE123813_bcc_scRNA_counts.txt.gz",
    sep="\t",
    index_col=0
)

# transpose counts (cells x genes)
adata = sc.AnnData(counts.T)

# set cell IDs in metadata
metadata = metadata.set_index(metadata.columns[0])

# align metadata to adata cells
metadata = metadata.loc[adata.obs_names]

# assign metadata
adata.obs = metadata

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
adata.obs['celltype'] = adata.obs['cluster']

In [ ]:
adata.obs['celltype'].unique()

In [ ]:
cd4_lineage = [
    "CD4_T_cells",
    "Tregs"
]
cd8_lineage = [
    "CD8_mem_T_cells",
    "CD8_ex_T_cells",
    "CD8_act_T_cells"
]
t_cells = [
    "CD4_T_cells",
    "Tregs",
    "CD8_mem_T_cells",
    "CD8_ex_T_cells",
    "CD8_act_T_cells",
   #  "Tcell_prolif"   les podem treure, no son res.
]

In [ ]:
adata_t = adata[adata.obs["celltype"].isin(cd8_lineage)].copy()

In [ ]:
sc.pp.filter_genes(adata_t, min_cells=10)

In [ ]:
sc.pp.normalize_total(adata_t)
sc.pp.log1p(adata_t)
sc.pp.highly_variable_genes(adata_t)
sc.pp.pca(adata_t)

In [ ]:
sc.pp.neighbors(adata_t)
sc.tl.leiden(adata_t, flavor='igraph', n_iterations=2)
sc.tl.umap(adata_t)


In [ ]:
sc.tl.umap(adata_t, min_dist=0.3)


In [ ]:
sc.pl.umap(adata_t, color=["leiden", "celltype", "patient", "treatment"])

In [ ]:
adata_t

In [ ]:
adata_cd4 = adata[adata.obs["celltype"].isin(cd4_lineage)].copy()

In [ ]:
sc.pp.filter_genes(adata_cd4, min_cells=10)

In [ ]:
sc.pp.normalize_total(adata_cd4)
sc.pp.log1p(adata_cd4)
sc.pp.highly_variable_genes(adata_cd4)
sc.pp.pca(adata_cd4)

In [ ]:
sc.pp.neighbors(adata_cd4)
sc.tl.leiden(adata_cd4, flavor='igraph', n_iterations=2)
sc.tl.umap(adata_cd4)


In [ ]:
sc.pl.umap(adata_cd4, color=["leiden", "celltype", "patient", "treatment"])

In [ ]:
%%time
import os
import scanpy as sc
import scvi
import seaborn as sns
import torch
from rich import print


os.environ["CUDA_VISIBLE_DEVICES"] = ""  # make to not see the GPU
ad = adata_t.copy()
scvi.model.SCVI.setup_anndata(ad, layer="counts", batch_key="patient")
model = scvi.model.SCVI(ad, n_layers=2, n_latent=30, gene_likelihood="nb")
model.train(accelerator="cpu", devices=1)

SCVI_LATENT_KEY = "X_scVI"
ad.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

sc.pp.neighbors(ad, use_rep=SCVI_LATENT_KEY)
#sc.tl.leiden(ad, resolution=0.7)
sc.tl.umap(ad) # we can change min_dist?!?!

sc.pl.umap(ad, color=["leiden", "celltype", "patient", "treatment"], title=label)

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    model,
    adata=ad,
    labels_key="celltype",
    unlabeled_category="Unknown",
)

In [ ]:
%%time
scanvi_model.train(max_epochs=20, n_samples_per_label=100)

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"
ad.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(ad)

In [ ]:
sc.pp.neighbors(ad, use_rep=SCANVI_LATENT_KEY)
sc.tl.umap(ad)

In [ ]:
sc.pl.umap(ad, color=["leiden", "celltype", "patient", "treatment"], title=label)

In [ ]:
adata_cd4 = adata[adata.obs["celltype"].isin(cd4_lineage)].copy()

In [ ]:
adata_cd4

In [ ]:
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=2000) #, subset=True)
#ad = ad[:, ad.var["highly_variable"]].copy()
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=16)
sc.pp.neighbors(adata, n_pcs=16)
sc.tl.leiden(adata, resolution=1, flavor="igraph", n_iterations=2)
sc.tl.umap(adata) # we can change min_dist?!?!



In [ ]:
sc.pl.umap(adata, color=["leiden", "celltype", "patient"])

In [ ]:
gold = pd.read_csv("/home/roger/Baixades/DEGs_nkts_vs_T.csv", index_col=0)
names = gold.head(50)['names'].to_list()
sc.tl.score_genes(adata, gene_list=names, score_name="nkt")

In [ ]:
sc.pl.umap(adata, color=['leiden', 'celltype', 'nkt', 'TRAV10', 'TRAV1-2'])

In [ ]:
sc.pl.violin(adata, keys=['nkt', 'TRAV10'], groupby='leiden')